# Lab 1 — Polynomial LSMC for an openIRM proxy model

**Lausanne actuarial workshop · 12 August 2026**  
**Mark-Oliver Wolf**  
**Revision 3 · verified 9 August 2026**

**Suggested time:** 15 minutes

In this lab, we will:

1. turn nested-simulation output into regression targets;
2. construct polynomial bases with deliberately selected interactions;
3. compare accuracy, basis size, and computation time;
4. evaluate against 1,000 benchmark states, each based on 10,000 inner simulations.

The notebook runs from top to bottom without changes. During the exercise, edit only the cell marked **YOUR TURN**, then rerun the cells below it.


## 1. Load the lab tools and data

To keep the exercise readable, technical code for loading, checking, fitting, evaluating, and plotting is stored in `lab1_utils.py`. You do not need to open or edit that file during the lab.

The notebook itself contains only the modelling choices and the calls that show their results. The external packages remain NumPy, pandas, Matplotlib, and scikit-learn.


In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

In [ ]:
import wednesday_lab1_utils as lab1

data = lab1.load_data()
lab1.data_summary(data)

## 2. From inner outcomes to conditional-expectation targets

For outer state $X_i$, `train_Z` contains $m=54$ conditionally simulated outcomes

$$Z_{i,1},\ldots,Z_{i,m}.$$

The proxy target is the conditional expectation

$$\mu(x)=\mathbb{E}[Z\mid X=x],$$

estimated at each training state by

$$\bar Z_i=\frac{1}{m}\sum_{j=1}^m Z_{i,j}.$$

Each row of `test_Y` uses 10,000 inner simulations and is therefore much closer to $\mu(X)$ than a training mean. Monetary values are expressed in **billions of euros** for readability and numerical conditioning.


In [ ]:
lab1.plot_training_targets(data)


### Why averaging is appropriate here

With the same number $m$ of inner outcomes at every outer state, fitting ordinary least squares to all repeated pairs $(X_i,Z_{i,j})$ gives the same coefficients as fitting to $(X_i,\bar Z_i)$:

$$
\sum_{i=1}^{n}\sum_{j=1}^{m}\bigl(Z_{i,j}-f(X_i)\bigr)^2
= C + m\sum_{i=1}^{n}\bigl(\bar Z_i-f(X_i)\bigr)^2,
$$

where $C$ does not depend on $f$. Averaging reduces $12{,}500\times54=675{,}000$ response rows to 12,500 regression targets without changing the OLS minimizer.

If the inner-simulation count differed between states, the aggregated regression would need weights proportional to those counts to retain this equivalence.


## 3. Polynomial LSMC: degree and interaction choice

LSMC chooses basis functions $\phi_1,\ldots,\phi_K$ and estimates

$$\widehat\mu(x)=\sum_{k=1}^{K}\widehat\beta_k\phi_k(x)$$

by least squares. An exhaustive polynomial basis contains every monomial up to a total degree. With $d$ risk factors, it contains

$$\binom{d+p}{p}-1$$

non-constant terms up to degree $p$. For 10 factors, degree 3 gives 285 terms; for 100 factors, it already gives 176,850 terms.

A practical alternative is to include univariate powers for every factor but add only interactions that are economically plausible or empirically supported. The lab standardizes the risk factors before constructing either basis.


## YOUR TURN — choose powers and interactions

Edit only the next cell. Use the risk-factor names exactly as written below:

| Available risk factor | Meaning |
|---|---|
| `initial_x` | Starting value first interest rate subprocess |
| `initial_y` | Starting value second interest rate subprocess |
| `yield_curve_pc1_risk_factor` | Roughly level of yield curve |
| `yield_curve_pc2_risk_factor` | Roughly slope of yield curve |
| `yield_curve_pc3_risk_factor` | Roughly curvature of yield curve |
| `stock_vola_risk_factor` | Volatility of stock process |
| `credit_default_rf` | Influences amount of defaulted bonds |
| `mortality_risk_factor` | Mortality of policyholders |
| `base_lapse_risk_factor` | Base lapse rate of contracts |
| `mass_lapse_risk_factor` | Mass lapse rate of contracts |

- `MAIN_EFFECT_DEGREE = 2` includes $x_j$ and $x_j^2$ for every risk factor.
- Each tuple in `SELECTED_INTERACTIONS` is multiplied into one additional basis term.
- A pair such as `("initial_x", "initial_y")` gives $x_1x_2$.
- A triple gives a three-factor interaction. Repeating one name, for example `("initial_x", "initial_x", "initial_y")`, gives $x_1^2x_2$.
- `ALL_TERMS_DEGREES` adds exhaustive reference models. Add degree 5 only if time permits.

Start by deleting and adding interactions. Predict how each change should affect basis size, runtime, and benchmark error.


In [ ]:
# YOUR TURN: change only the settings in this cell.
MAIN_EFFECT_DEGREE = 3

SELECTED_INTERACTIONS = [
    ("initial_x", "initial_y"),
    ("yield_curve_pc1_risk_factor", "yield_curve_pc2_risk_factor"),
    ("yield_curve_pc1_risk_factor", "yield_curve_pc3_risk_factor"),
    ("yield_curve_pc1_risk_factor", "yield_curve_pc2_risk_factor", "yield_curve_pc3_risk_factor"),
    ("stock_vola_risk_factor", "credit_default_rf"),
    ("base_lapse_risk_factor", "mass_lapse_risk_factor"),
    ("stock_vola_risk_factor", "yield_curve_pc1_risk_factor"),
    ("stock_vola_risk_factor", "yield_curve_pc2_risk_factor"),
]

# Exhaustive reference models, for example: (2,), (2, 3), or (2, 3, 4).
ALL_TERMS_DEGREES = (2, 3, 4)


## 4. Run the experiment

The next cell fits the compact specifications and the selected exhaustive reference models. Lower test RMSE is better. Runtime is machine-dependent; its relative change is the useful observation.


In [ ]:
experiment = lab1.run_experiment(
    data, MAIN_EFFECT_DEGREE, SELECTED_INTERACTIONS, ALL_TERMS_DEGREES
)
experiment.results.round(4)


In [ ]:
lab1.plot_model_comparison(experiment)


In [ ]:
lab1.plot_best_prediction(experiment)


## Debrief

Discuss with your neighbour:

1. Which selected interactions changed the benchmark error relative to main effects only?
2. How close did the compact selected basis come to the exhaustive basis?
3. How did basis size and runtime change when you added all degree-3 terms?
4. What subject-matter arguments could guide interaction selection in an internal model?
5. Why should the test benchmark not be reused indefinitely for model selection?
